In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from collections import deque, namedtuple
import random

# Constants
CAPITAL = 100000
MEMORY_LEN = 10000
BATCH_SIZE = 64
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 200
LEARNING_RATE = 1e-3
NUM_EPOCHS = 50

# Define the environment
class SingleAssetTradingEnvironment:
    def __init__(self, data, signals, initial_money=CAPITAL):
        self.data = data
        self.signals = signals
        self.initial_money = initial_money
        self.reset()

    def reset(self):
        self.pointer = 0
        self.capital = self.initial_money
        self.holding = 0
        self.done = False
        return self.get_state()

    def step(self, action):
        current_price = self.data.iloc[self.pointer]['close']
        if action == 1:  # Buy
            if self.capital >= current_price:
                self.holding += self.capital // current_price
                self.capital -= self.holding * current_price
        elif action == 0:  # Hold (No action)
            pass
        elif action == 2:  # Sell
            if self.holding > 0:
                self.capital += self.holding * current_price
                self.holding = 0

        self.pointer += 1
        if self.pointer >= len(self.data) - 1:
            self.done = True

        next_state = self.get_state()
        reward = self.calculate_reward()
        return next_state, reward, self.done

    def get_state(self):
        return np.array([self.capital, self.holding, self.data.iloc[self.pointer]['close'], self.signals[self.pointer]])

    def calculate_reward(self):
        return self.capital + self.holding * self.data.iloc[self.pointer]['close'] - self.initial_money

# Define the DQN model
class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        self.fc = nn.Linear(input_size, 2 * input_size)
        self.relu = nn.ReLU()
        self.out = nn.Linear(2 * input_size, output_size)  # Fix the output size

    def forward(self, x):
        x = self.relu(self.fc(x))
        return self.out(x)

# Define the DQNAgent
class DQNAgent:
    def __init__(self, input_size, output_size):
        self.policy_net = DQN(input_size, output_size)
        self.target_net = DQN(input_size, output_size)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.memory = deque(maxlen=MEMORY_LEN)
        self.steps_done = 0

    def select_action(self, state, eps):
        self.steps_done += 1
        if random.random() > eps:
            with torch.no_grad():
                action = self.policy_net(torch.Tensor(state)).argmax().item()
        else:
            action = random.choice([2, 0, 1])  # Updated action space
        return action

    def store_transition(self, state, action, next_state, reward, done):
        self.memory.append((state, action, next_state, reward, done))

    def optimize_model(self):
        if len(self.memory) < BATCH_SIZE:
            return

        transitions = random.sample(self.memory, BATCH_SIZE)
        batch = Transition(*zip(*transitions))

        non_final_mask = torch.tensor(tuple(map(lambda s: s is not None, batch.next_state)), dtype=torch.bool)
        non_final_next_states = torch.stack([torch.Tensor(s) for s in batch.next_state if s is not None])

        state_batch = torch.stack([torch.Tensor(s) for s in batch.state])
        action_batch = torch.LongTensor(batch.action).view(-1, 1)
        reward_batch = torch.Tensor(batch.reward)
        next_state_values = torch.zeros(BATCH_SIZE)

        # Compute Q(s_t, a)
        state_action_values = self.policy_net(state_batch).gather(1, action_batch)

        # Compute V(s_{t+1}) for all next states.
        next_state_values[non_final_mask] = self.target_net(non_final_next_states).max(1)[0].detach()

        # Compute the expected Q values
        expected_state_action_values = reward_batch + GAMMA * next_state_values

        # Compute Huber loss
        loss = F.smooth_l1_loss(state_action_values, expected_state_action_values.unsqueeze(1))

        # Optimize the model
        self.optimizer.zero_grad()
        loss.backward()
        for param in self.policy_net.parameters():
            param.grad.data.clamp_(-1, 1)
        self.optimizer.step()

# Define the Transition namedtuple
Transition = namedtuple("Transition", ["state", "action", "next_state", "reward", "done"])

# Function to create trading signals
def create_signal_labels(timeseries):
    signals = []
    curr_sig = 0
    closed = True
    for i in range(len(timeseries) - 1):
        # set 1
        if timeseries[i + 1] > timeseries[i]:
            if closed:
                signals.append(1)
                closed = False
                curr_sig = 1
            else:
                if curr_sig == -1:
                    closed = True
                    curr_sig = 0
                    signals.append(1)
                else:
                    signals.append(0)
        # set -1
        if timeseries[i + 1] < timeseries[i]:
            if closed:
                signals.append(-1)
                closed = False
                curr_sig = -1
            else:
                if curr_sig == 1:
                    closed = True
                    curr_sig = 0
                    signals.append(-1)
                else:
                    signals.append(0)
    signals.append(0)

    # Ensure the length matches the input timeseries
    while len(signals) < len(timeseries):
        signals.append(0)

    return signals

# Load data and generate signals
data = pd.read_csv("btcusdt_1h.csv")
closing_prices = data['close'].values
signals = create_signal_labels(closing_prices)

# Create environment and agent
env = SingleAssetTradingEnvironment(data, signals)
agent = DQNAgent(input_size=4, output_size=3)

# # Training loop
# for epoch in range(NUM_EPOCHS):
#     state = env.reset()
#     total_reward = 0
#     eps = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * agent.steps_done / EPS_DECAY)

#     for step in range(len(data)):
#         action = agent.select_action(state, eps)
#         next_state, reward, done = env.step(action)
#         agent.store_transition(state, action, next_state, reward, done)
#         agent.optimize_model()

#         state = next_state
#         total_reward += reward

#         if done:
#             break

#     print(f"Epoch {epoch + 1}, Total Reward: {total_reward}")

# Training loop
cumulative_returns = []

for epoch in range(NUM_EPOCHS):
    state = env.reset()
    total_reward = 0
    portfolio_values = []  # Store portfolio values at each step
    eps = EPS_END + (EPS_START - EPS_END) * np.exp(-1. * agent.steps_done / EPS_DECAY)

    for step in range(len(data)):
        action = agent.select_action(state, eps)
        next_state, reward, done = env.step(action)
        agent.store_transition(state, action, next_state, reward, done)
        agent.optimize_model()

        state = next_state
        total_reward += reward
        portfolio_values.append(env.capital + env.holding * env.data.iloc[env.pointer]['close'])

        if done:
            break

    cumulative_returns.append((portfolio_values[-1] - CAPITAL) / CAPITAL)

    print(f"Epoch {epoch + 1}, Total Reward: {total_reward}")

# Calculate cumulative returns
cumulative_returns = np.array(cumulative_returns)
cumulative_returns_percentage = (cumulative_returns * 100).tolist()
print(f"Cumulative Returns: {cumulative_returns_percentage}%")


Epoch 1, Total Reward: -3267655054.829173
Epoch 2, Total Reward: -2228315361.009412
Epoch 3, Total Reward: -462847296.5699721
Epoch 4, Total Reward: 9831984311.729906
Epoch 5, Total Reward: -3026030368.931768
Epoch 6, Total Reward: -3186980160.928762
Epoch 7, Total Reward: -2360648265.420356
Epoch 8, Total Reward: -1128160753.3101301
Epoch 9, Total Reward: -2405874929.330086
Epoch 10, Total Reward: -3181804260.151459
Epoch 11, Total Reward: -2359652637.9098415
Epoch 12, Total Reward: -949684861.9800607
Epoch 13, Total Reward: 366034059.17990553
Epoch 14, Total Reward: 1896617584.7702951
Epoch 15, Total Reward: -3059357978.739903
Epoch 16, Total Reward: -3175187224.3193336
Epoch 17, Total Reward: -3202135744.338078
Epoch 18, Total Reward: -1776452357.03012
Epoch 19, Total Reward: -1094901154.910181
Epoch 20, Total Reward: 1689651890.019433
Epoch 21, Total Reward: 40718423.79999912
Epoch 22, Total Reward: -2561256628.091006
Epoch 23, Total Reward: -2956674998.010586
Epoch 24, Total Rewar